# Blob to Bronze â€” Transactions Auto Loader

Reads CSV files arriving at the configured volume path using Databricks Auto Loader and writes them to the Delta table `{catalog}.blob_bronze.transactions`.

In [0]:
# â”€â”€ Auto Loader: CSV â†’ Delta bronze table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
dbutils.widgets.text("catalog", "retail_q_dab_dev", "Catalog")
catalog = dbutils.widgets.get("catalog")

source_path       = f"/Volumes/{catalog}/volumes/blob_source/transactions_source/"
schema_location   = f"/Volumes/{catalog}/volumes/blob_source/_schemas/"
checkpoint_path   = f"/Volumes/{catalog}/volumes/blob_source/_checkpoints/"
target_table      = f"{catalog}.blob_bronze.transactions"

# Define schema explicitly â€” avoids a full data scan for inference in streaming
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

csv_schema = StructType([
    StructField("transaction_id",       StringType(),  True),
    StructField("opportunity_name",     StringType(),  True),
    StructField("product_id",            StringType(),  True),
    StructField("store_id",              StringType(),  True),
    StructField("quantity",              IntegerType(), True),
    StructField("selling_price",         IntegerType(), True),
    StructField("discount_amount",       IntegerType(), True),
    StructField("transaction_timestamp", StringType(),  True),
    StructField("payment_mode",          StringType(),  True),
    StructField("sales_channel",         StringType(),  True),
])

# Read CSV files incrementally via Auto Loader
raw_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaHints", 
                "transaction_id string, opportunity_name string, "
                "product_id string, store_id string, "
                "quantity int, selling_price int, discount_amount int, "
                "transaction_timestamp string, payment_mode string, "
                "sales_channel string")
        .option("header", "true")
        .option("ignoreLeadingWhiteSpace", "true")
        .option("ignoreTrailingWhiteSpace", "true")
        .option("mode", "PERMISSIVE")
        .schema(csv_schema)
        .load(source_path)
)

# Add bronze-layer metadata columns
from pyspark.sql.functions import current_timestamp, col

bronze_df = (
    raw_df
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
)

# Write to the Delta target table via foreachBatch (append mode)
(
    bronze_df.writeStream
        .foreachBatch(lambda batch_df, batch_id: (
            batch_df.write
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(target_table)
        ))
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
)

print("Auto Loader stream started. Use the next cell to verify loaded data.")

In [0]:
# â”€â”€ Verify the data landed in the bronze table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
spark.sql(f"SELECT COUNT(*) AS row_count FROM {target_table}").display()
spark.sql(f"SELECT * FROM {target_table} LIMIT 10").display()

In [0]:
spark.sql(f"SELECT * FROM {target_table}").display()